# Part 1: anchored stopping-reason classifier

The current model has a **user-reported Kaggle sample-wise F1 of 0.89636**. Its final fit uses **235 verified labels** (157 training + 78 validation) and the **1,014 originally masked training rows**. Kaggle test inputs enter prediction only.

This notebook explains the fixed recipe, performs one fresh final fit, reproduces the exact scored CSV in memory, and verifies the saved grouped comparison. **It does not rerun the 80 CV fits.** The [testing policy](TESTING_POLICY.md) and full-replay command below specify that separate evaluation.

[Current overview](README.md) · [Exact scored result](submissions/results/anchored-trainval235.json) · [Research index](RESEARCH_INDEX.html) · [Historical entry notebooks](research/historical_notebooks/README.md)

The original notebooks and earlier outputs are archived. Ani's original .89423 source and later LDA revision are retained separately; the original score must not be assigned to the later revision.

In [1]:
from pathlib import Path
import hashlib
import json
import sys
import warnings

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "anchored_solution.py").is_file():
        PART1 = folder
        break
    if (folder / "part1/anchored_solution.py").is_file():
        PART1 = folder / "part1"
        break
else:
    raise RuntimeError("Run from the repository root or its part1 directory.")
if str(PART1) not in sys.path:
    sys.path.insert(0, str(PART1))

REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
FEATURES = [f"emb_{i}" for i in range(150)]
CANDIDATE = PART1 / "experiments/paired_improvement/candidate"
COMPARISON = PART1 / "experiments/current_leaders_grouped_v1"
frozen = json.loads((CANDIDATE / "manifest.json").read_text())
result = json.loads((PART1 / "submissions/results/anchored-trainval235.json").read_text())

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

protected = {}
for name, digest in frozen["input_sha256"].items():
    protected[PART1 / "data" / name] = digest
for name, digest in frozen["source_sha256"].items():
    protected[PART1 / name] = digest
for name, digest in frozen["artifact_sha256"].items():
    protected[CANDIDATE / name] = digest
protected[CANDIDATE / "manifest.json"] = result["frozen_recipe_sha256"]
SCORED_CSV = PART1.parent / result["submission_file"]
protected[SCORED_CSV] = result["submission_sha256"]
for path, digest in protected.items():
    assert sha256(path) == digest, path.name

display(pd.DataFrame({
    "recorded": {k: frozen["versions"][k] for k in ["python", "numpy", "pandas", "sklearn"]},
    "this_run": {"python": sys.version.split()[0], "numpy": np.__version__,
                 "pandas": pd.__version__, "sklearn": sklearn.__version__},
}))

,recorded,this_run
python,3.12.13,3.12.13
numpy,2.5.2,2.5.2
pandas,3.0.5,3.0.5
sklearn,1.9.0,1.9.0


## Data and feature choice

The assignment provides 150 numeric embeddings of the stopping-reason text. These directly describe the prediction target, so the selected classifier uses those embeddings. IDs serve alignment and audits only. Start year is used in chronological stress tests, not as a predictor in this recipe.

Earlier experiments tested eligibility text, diseases, pathways, drug features, and temporal features. Those tests did not produce a confirmed replacement for the selected model; many used older parents and evaluation protocols. Their complete recipe and score summaries remain in the [older-study catalog](research/legacy_retention_v1/README.md) and [LDA research catalog](research/lda_ensemble/experiments/lda_research_checkpoint_v1/index.html).

A value of `-100` is an unknown reason label, not a negative label. Preserve all verified labels. For the final fit, validation joins the fitting pool; the old 78-row validation score is therefore **not independent validation of this final fitted artifact**.

In [2]:
train = pd.read_csv(PART1 / "data/part1_train.csv")
validation = pd.read_csv(PART1 / "data/part1_val.csv")
fit_frame = pd.concat([train, validation], ignore_index=True)  # frozen original row order
known = fit_frame[REASONS].isin([0, 1]).all(axis=1).to_numpy()
masked = fit_frame[REASONS].eq(-100).all(axis=1).to_numpy()
assert (known | masked).all()
assert fit_frame.ID.is_unique
assert (len(train), len(validation), int(known.sum()), int(masked.sum())) == (1171, 78, 235, 1014)
assert set(train.ID).isdisjoint(validation.ID)

display(pd.DataFrame({
    "population": ["Verified original training", "Originally masked training", "Verified validation"],
    "rows": [157, 1014, 78],
    "final_fit_role": ["Gold labels", "Unsupervised structure and generated-label candidates", "Gold labels"],
}))

,population,rows,final_fit_role
0,Verified original training,157,Gold labels
1,Originally masked training,1014,Unsupervised structure and generated-label can...
2,Verified validation,78,Gold labels


## Frozen model

1. Standardize the 150 embeddings on the permitted fitting rows. Fit PCA with 30 components and a 30-component diagonal GMM to support missing-label generation.
2. Fit a joint logistic classifier (`C=0.1`), shrinkage LDA, and cosine 10-neighbor predictor. The final blend is **50% logistic + 30% neighbors + 20% LDA**.
3. Derive logistic class weights from verified labels and hold them fixed. Generated rows receive one quarter of the corresponding verified-label weight in logistic fitting. A verified row with two reasons contributes to both; neighbor votes use binary reason indicators.
4. Run two generated-label passes. Teaching scores combine 95% classifier blend and 5% GMM-derived reason rates; accept a row when its maximum teaching score is at least 0.8. Rebuild each round from verified labels plus accepted generated rows.
5. Predict every reason scoring at least 0.3, with the highest-scoring reason included to guarantee a nonempty prediction. These scores are selection scores, not calibrated marginal probabilities.

Every learned step must be fitted within the permitted training partition during CV. The complete implementation below is copied unchanged from [anchored_solution.py](anchored_solution.py).

In [3]:
"""Current Ani with three small changes: anchored weights, binary votes, weaker GMM.

Fit receives only permitted rows, in original CSV order. Fully missing labels
are -100; query/test rows must never be passed to fit. Prediction uses only the
150 stopping-reason embedding columns, ordered emb_0 through emb_149.
"""
import numpy as np
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, normalize


def select_reasons(scores):
    result=(scores>=.3).astype(int)
    result[np.arange(len(scores)),scores.argmax(axis=1)]=1
    return result


class AnchoredReasonClassifier:
    def fit(self, embeddings, labels):
        X,Y=np.asarray(embeddings,float),np.asarray(labels,int)
        if X.ndim!=2 or X.shape[1]!=150 or Y.shape!=(len(X),5):
            raise ValueError('Expected 150 embeddings and five labels per row.')
        known=np.isin(Y,[0,1]).all(axis=1)
        unknown=(Y==-100).all(axis=1)
        if not (known|unknown).all() or (Y[known].sum(axis=1)==0).any() or (Y[known].sum(axis=0)==0).any():
            raise ValueError('Use complete nonempty known labels or five -100 values; all five reasons must be represented.')
        self.scaler_=StandardScaler().fit(X)
        Z=self.scaler_.transform(X)
        self.pca_=PCA(30,random_state=0).fit(Z)
        self.gmm_=GaussianMixture(30,covariance_type='diag',n_init=3,random_state=0).fit(self.pca_.transform(Z))
        assert self.gmm_.converged_
        K,V=Z[known],Z[unknown];targets=Y[known]
        Rk=self.gmm_.predict_proba(self.pca_.transform(K))
        Ru=self.gmm_.predict_proba(self.pca_.transform(V))
        counts=(targets/targets.sum(axis=1,keepdims=True)).sum(axis=0)
        self.class_weights_=counts.mean()/counts
        Za,Ya,Ra=K,targets,Rk
        self.round_audit_=[]
        for step in range(3):
            row,reason=np.nonzero(Ya)
            weights=self.class_weights_[reason]/Ya.sum(axis=1)[row]
            weights*=np.where(row<len(K),1.,.25)
            self.logistic_=LogisticRegression(C=.1,max_iter=3000,tol=1e-4).fit(Za[row],reason,sample_weight=weights)
            self.lda_=LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto',priors=np.ones(5)/5).fit(Za[row],reason)
            self.neighbor_X_=normalize(Za)
            self.neighbor_Y_=Ya.copy()
            self.round_audit_.append(dict(step=step,known=len(K),generated=len(Za)-len(K),
                                          known_weight=float(weights[row<len(K)].sum()),
                                          generated_weight=float(weights[row>=len(K)].sum())))
            if step==2:break
            T=Ya/Ya.sum(axis=1,keepdims=True)
            rates=(Ra.T@T+T.mean(axis=0))/(Ra.sum(axis=0)[:,None]+1)
            scores=.95*self._scores(V)+.05*(Ru@rates)
            self.generated_mask_=scores.max(axis=1)>=.8
            self.generated_labels_=select_reasons(scores[self.generated_mask_])
            Za=np.vstack([K,V[self.generated_mask_]])
            Ya=np.vstack([targets,self.generated_labels_])
            Ra=np.vstack([Rk,Ru[self.generated_mask_]])
        return self

    def _scores(self,Z):
        logistic=np.zeros((len(Z),5))
        logistic[:,self.logistic_.classes_]=self.logistic_.predict_proba(Z)
        similarity=normalize(Z)@self.neighbor_X_.T
        neighbors=np.argsort(-similarity,axis=1)[:,:10]
        weights=np.take_along_axis(similarity,neighbors,axis=1).clip(0)**2
        local=(weights[:,:,None]*self.neighbor_Y_[neighbors]).sum(axis=1)
        local/=weights.sum(axis=1,keepdims=True)+1e-9
        lda=np.zeros_like(logistic)
        lda[:,self.lda_.classes_]=self.lda_.predict_proba(Z)
        return (1-.3-.2)*logistic+.3*local+.2*lda

    def predict_proba(self, embeddings):
        """Selection scores, not calibrated marginal probabilities."""
        return self._scores(self.scaler_.transform(np.asarray(embeddings,float)))

    def predict(self, embeddings):
        return select_reasons(self.predict_proba(embeddings))


## Refit on all 235 verified labels

This executes the selected recipe once. It does not change settings or select a model. The final training pool contains 1,249 rows; test rows have not been passed to any fitting step.

In [4]:
with threadpool_limits(limits=1), warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    model = AnchoredReasonClassifier().fit(fit_frame[FEATURES], fit_frame[REASONS])

audit = pd.DataFrame(model.round_audit_)
expected_audit = pd.DataFrame(frozen["final_fit_audit"])
pd.testing.assert_frame_equal(audit, expected_audit, check_exact=False, atol=1e-7, rtol=0)
assert audit.generated.tolist() == [0, 685, 757]
assert audit.known.tolist() == [235, 235, 235]
display(audit)

,step,known,generated,known_weight,generated_weight
0,0,235,0,235.0,0.000000
1,1,235,685,235.0,102.796878
2,2,235,757,235.0,119.526927


## Reproduce the scored submission

Test data is opened for prediction after fitting. The CSV is assembled in the supplied sample order and must match the recorded submission **byte for byte**. The reported Kaggle score is historical user feedback; this notebook does not query or upload to Kaggle.

In [5]:
test = pd.read_csv(PART1 / "data/part1_test.csv")
template = pd.read_csv(PART1 / "data/part1_sample_submission.csv")
assert len(test) == 312 and test.ID.is_unique
assert set(test.ID).isdisjoint(fit_frame.ID)
with threadpool_limits(limits=1):
    scores = model.predict_proba(test[FEATURES])
    predictions = model.predict(test[FEATURES])

recorded_scores = pd.read_csv(CANDIDATE / "test_scores.csv").set_index("ID").loc[test.ID, REASONS]
np.testing.assert_allclose(scores, recorded_scores.to_numpy(), atol=1e-7, rtol=0)
labels = [" ".join(reason for reason, selected in zip(REASONS, row) if selected)
          for row in predictions]
submission = pd.DataFrame({"ID": test.ID, "Predicted": labels}).set_index("ID").loc[template.ID].reset_index()
assert submission.ID.is_unique and submission.Predicted.str.len().gt(0).all()
submission_bytes = submission.to_csv(index=False).encode("utf-8")
assert hashlib.sha256(submission_bytes).hexdigest() == result["submission_sha256"]
assert submission_bytes == SCORED_CSV.read_bytes()
display(submission.head())
print("Fresh 235-label fit reproduced all 312 scored predictions and the exact CSV hash.")

,ID,Predicted
0,TR00756,Efficacy
1,TR05267,Safety
2,TR10354,Safety
3,TR01930,Recruitment
4,TR01046,Other


Fresh 235-label fit reproduced all 312 scored predictions and the exact CSV hash.


## Generated training labels

The final classifier accepted **757** generated rows during fitting. For the course's augmented-training deliverable, the fitted model also predicts labels for **all 1,014 originally masked rows**. These are different counts and purposes. The augmented export preserves every originally verified training label and identifies generated labels explicitly.

In [6]:
training_masked = train[REASONS].eq(-100).all(axis=1)
with threadpool_limits(limits=1):
    generated = model.predict(train.loc[training_masked, FEATURES])
augmented = train.copy()
augmented.loc[training_masked, REASONS] = generated
augmented["label_origin"] = np.where(training_masked, "model_generated", "verified_training")
np.testing.assert_array_equal(augmented.loc[~training_masked, REASONS], train.loc[~training_masked, REASONS])
recorded_augmented = pd.read_csv(CANDIDATE / "part1_train_augmented.csv").set_index("ID")
np.testing.assert_array_equal(augmented[REASONS], recorded_augmented.loc[augmented.ID, REASONS])
assert augmented[REASONS].isin([0, 1]).all().all()
display(augmented.groupby("label_origin").size().rename("rows").to_frame())

,rows
label_origin,
model_generated,1014
verified_training,157


## Apples-to-apples evaluation

The current benchmark uses eight saved repeats of five grouped folds on each pool: **Train157 (40 fits)** and **Pooled235 (40 fits)**. It adds the original 78-row validation fit and three fixed chronological windows containing 98 held-out rows. Fit preprocessing and pseudo-labeling inside each fold, purge masked duplicates of held-out embedding groups, and exclude held-out/test inputs from fitting.

The table below reports saved, independently checked results. Anchored is effectively tied with the LDA ensemble on CV: slightly higher on Train157 and slightly lower on Pooled235. Its original validation score is lower, while pooled chronology is higher. The final 0.89636 is a reported Kaggle result, not a CV score.

These reused development folds do not provide an untouched holdout. The [testing policy](TESTING_POLICY.md) defines the exact assignments, population SD, paired comparisons, advancement rules, and distinctions from older ungrouped benchmarks.

In [7]:
from current_leaders_local import verify as verify_current_leaders
saved_evidence = verify_current_leaders()
assert saved_evidence["passed"] and saved_evidence["models_fitted"] == 0
print({key: saved_evidence[key] for key in [
    "independently_rescored_complete_repeats", "ani_control_exact_predictions", "scored_archive_identical"
]})
comparison = pd.read_csv(COMPARISON / "comparison.csv").set_index("name")
show = comparison.loc[["anchored_candidate", "lda_ensemble", "current_ani_lda"],
                      ["train", "pooled", "validation", "chronology98", "reported_kaggle"]]
display(show.rename(columns={"train": "Train157 CV", "pooled": "Pooled235 CV",
                             "validation": "Validation78", "chronology98": "Chronology98",
                             "reported_kaggle": "Reported Kaggle"}).round(6))

{'independently_rescored_complete_repeats': 128, 'ani_control_exact_predictions': 80, 'scored_archive_identical': True}


,Train157 CV,Pooled235 CV,Validation78,Chronology98,Reported Kaggle
name,,,,,
anchored_candidate,0.932059,0.921365,0.891026,0.884354,0.89636
lda_ensemble,0.931794,0.921631,0.905983,0.877551,0.89401
current_ani_lda,0.928079,0.915426,0.871795,0.874150,NaN


To explicitly replay the full comparison, run from the repository root with a **new** output directory:

```sh
python part1/current_leaders_local.py replay --output /tmp/part1-leaders-replay
```

That command runs 80 anchored CV fits, 80 current Ani control CV fits, and four anchored supplementary fits. Other comparison entries are rescored from retained predictions on the identical plans. Use the [research index](RESEARCH_INDEX.html) for individual archived study replays.

## Optional disposable export

The scored CSV and augmented data already exist in the frozen candidate directory. Set the flag below only to write this fresh replay to a new temporary directory. It never overwrites those original files or submits anything.

In [8]:
EXPORT_REPLAY = False
if EXPORT_REPLAY:
    import tempfile
    replay_dir = Path(tempfile.mkdtemp(prefix="part1-anchored-replay-"))
    (replay_dir / "part1_anchored_submission.csv").write_bytes(submission_bytes)
    augmented.to_csv(replay_dir / "part1_train_augmented.csv", index=False)
    print("Replay outputs:", replay_dir)

for path, digest in protected.items():
    assert sha256(path) == digest, path.name
NOTEBOOK_CHECKS = {
    "fresh_final_fits": 1,
    "verified_labels": int(known.sum()),
    "original_masked_rows": int(masked.sum()),
    "generated_rows_in_final_fit": int(model.generated_mask_.sum()),
    "test_predictions": len(submission),
    "submission_sha256": hashlib.sha256(submission_bytes).hexdigest(),
    "exact_scored_csv": True,
    "generated_label_export_matches": True,
    "frozen_files_unchanged": True,
    "saved_comparison_verified": saved_evidence["passed"],
    "full_cv_refitted_here": False,
}
print(json.dumps(NOTEBOOK_CHECKS, indent=2))

{
  "fresh_final_fits": 1,
  "verified_labels": 235,
  "original_masked_rows": 1014,
  "generated_rows_in_final_fit": 757,
  "test_predictions": 312,
  "submission_sha256": "1db6433dc5a5b900d440db16e5ef8375eb51dcf02d3f270217527d8b174319eb",
  "exact_scored_csv": true,
  "generated_label_export_matches": true,
  "frozen_files_unchanged": true,
  "saved_comparison_verified": true,
  "full_cv_refitted_here": false
}
